## 01 — ipyleaflet Intro

In the last lesson we used geojson.io to view and edit GeoJSON in the browser. This lesson brings the map **into the notebook**.

**ipyleaflet** is a Python library that wraps the Leaflet.js mapping library. It renders interactive maps directly inside Jupyter — you can pan, zoom, click features, and update the map programmatically from Python.

**ipywidgets** is the underlying widget framework that makes this possible. ipyleaflet's `Map`, markers, and layers are all widgets — objects that have live state and can respond to events. You do not need to understand all of ipywidgets to use ipyleaflet, but knowing it is there explains why the map is interactive rather than just a static image.

## The Widget Model

A regular matplotlib plot is static — once rendered, it is just an image. An ipyleaflet map is a **widget**: a live Python object linked to something rendered in the browser.

```
Python object          Browser display
──────────────         ───────────────
Map(zoom=10)    ←───►  interactive map tile
Marker(...)     ←───►  draggable pin
layer.add(...)  ←───►  updates the map in place
```

This means you can add layers, move markers, or change zoom programmatically **after** the map is already displayed — without re-running the cell.

## Your First Map

All ipyleaflet maps start with `Map`. The two required parameters are `center` and `zoom`.

`center` takes `[latitude, longitude]` — **note: this is lat/lon order**, the opposite of GeoJSON's `[lon, lat]`. This is a common source of confusion.

In [2]:
from ipyleaflet import Map

# Wichita Falls, TX
WICHITA_FALLS = (33.9137, -98.4934)  # (lat, lon) — ipyleaflet order

m = Map(center=WICHITA_FALLS, zoom=12)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Zoom Levels

Zoom is an integer roughly on this scale:

| Zoom | What you see |
|---|---|
| 1–3 | whole continents |
| 5–7 | country / state level |
| 10–12 | city level |
| 14–16 | neighborhood / street level |
| 18+ | individual buildings |

You can also zoom interactively with the scroll wheel or the `+`/`-` controls on the map.

## Adding a Default Marker

A `Marker` is the simplest feature — a pin at a location. Like the map itself, a marker is a widget: you add it to the map with `m.add(marker)`.

In [3]:
from ipyleaflet import Map, Marker

m = Map(center=WICHITA_FALLS, zoom=13)

# Marker location is also (lat, lon)
city_hall = Marker(location=(33.9137, -98.4934), title="Wichita Falls City Hall")
m.add(city_hall)

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Multiple Markers

You can add as many markers as you want. Build them in a loop and call `m.add()` for each one.

In [ ]:
from ipyleaflet import Map, Marker

m = Map(center=WICHITA_FALLS, zoom=12)

locations = [
    {"name": "Wichita Falls City Hall",   "lat": 33.9137, "lon": -98.4934},
    {"name": "Midwestern State University","lat": 33.8956, "lon": -98.5187},
    {"name": "Lake Wichita",               "lat": 33.8627, "lon": -98.5457},
    {"name": "Lucy Park",                  "lat": 33.9281, "lon": -98.5071},
]

for loc in locations:
    marker = Marker(location=(loc["lat"], loc["lon"]), title=loc["name"])
    m.add(marker)

m

## Custom Markers with AwesomeIcon

The default marker is a blue teardrop. `AwesomeIcon` lets you swap in any [Font Awesome](https://fontawesome.com/v4/icons/) icon and change the color.

Available `marker_color` values: `"red"`, `"blue"`, `"green"`, `"purple"`, `"orange"`, `"darkred"`, `"lightred"`, `"beige"`, `"darkblue"`, `"darkgreen"`, `"cadetblue"`, `"darkpurple"`, `"white"`, `"pink"`, `"lightblue"`, `"lightgreen"`, `"gray"`, `"lightgray"`, `"black"`

In [5]:
from ipyleaflet import Map, Marker, AwesomeIcon

m = Map(center=WICHITA_FALLS, zoom=12)

# university — book icon, dark blue
university_icon = AwesomeIcon(name="graduation-cap", marker_color="darkblue", icon_color="white")
university = Marker(
    location=(33.8956, -98.5187),
    icon=university_icon,
    title="Midwestern State University"
)

# park — tree icon, green
park_icon = AwesomeIcon(name="tree", marker_color="green", icon_color="white")
park = Marker(
    location=(33.9281, -98.5071),
    icon=park_icon,
    title="Lucy Park"
)

# lake — tint (water drop) icon, cadetblue
lake_icon = AwesomeIcon(name="tint", marker_color="cadetblue", icon_color="white")
lake = Marker(
    location=(33.8627, -98.5457),
    icon=lake_icon,
    title="Lake Wichita"
)

m.add(university)
m.add(park)
m.add(lake)

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Popup Messages

A `Popup` attaches an HTML label to a marker that appears when you click it. Use `ipywidgets.HTML` to define the content.

In [6]:
from ipyleaflet import Map, Marker, Popup
from ipywidgets import HTML

m = Map(center=WICHITA_FALLS, zoom=13)

marker = Marker(location=(33.9137, -98.4934))

popup_content = HTML(value="""
    <b>Wichita Falls, TX</b><br>
    Population: ~100,000<br>
    Founded: 1876
""")

marker.popup = popup_content
m.add(marker)

m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

## Changing the Tile Layer

The default background tiles are OpenStreetMap. ipyleaflet ships with several alternatives via `basemaps`.

In [7]:
from ipyleaflet import Map, basemaps

# satellite imagery
m = Map(
    center=WICHITA_FALLS,
    zoom=12,
    basemap=basemaps.Esri.WorldImagery
)
m

Map(center=[33.9137, -98.4934], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

Other useful basemaps:

```python
basemaps.OpenStreetMap.Mapnik      # default street map
basemaps.Esri.WorldImagery         # satellite
basemaps.Esri.NatGeoWorldMap       # National Geographic style
basemaps.CartoDB.Positron          # light/minimal
basemaps.CartoDB.DarkMatter        # dark mode
basemaps.OpenTopoMap               # topographic
```

## Exercise A

Pick 3 US cities you know. Look up their `(lat, lon)` coordinates and add a default `Marker` for each — with `title` set to the city name — on a zoom-5 map centered roughly over the US.

In [ ]:
from ipyleaflet import Map, Marker

m = Map(center=(39.5, -98.35), zoom=4)
for name, latlon in {
    "Wichita Falls": (33.9137, -98.4934),
    "Dallas": (32.7767, -96.7970),
    "Austin": (30.2672, -97.7431),
}.items():
    marker = Marker(location=latlon, title=name)
    m.add(marker)
m


## Exercise B

Take one of your markers from Exercise A and attach a `Popup` using `ipywidgets.HTML`. The popup should display the city name and one fact (population, founding year, or elevation) when clicked.

In [ ]:
from ipyleaflet import Map, Marker, Popup
from ipywidgets import HTML

m = Map(center=(33.9137, -98.4934), zoom=11)
marker = Marker(location=(33.9137, -98.4934), title="Wichita Falls")
marker.popup = Popup(
    location=marker.location,
    child=HTML("<b>Wichita Falls</b><br>Interactive popup example"),
    close_button=False,
    auto_close=False,
    close_on_escape_key=False,
)
m.add(marker)
m


```python
from ipyleaflet import Map, Marker, AwesomeIcon

m = Map(center=(33.9137, -98.4934), zoom=12)
points = [
    ("City Hall", (33.9137, -98.4934), "flag", "red"),
    ("MSU", (33.8763, -98.5204), "graduation-cap", "blue"),
    ("Lucy Park", (33.9296, -98.5000), "tree", "green"),
]
for name, location, icon_name, color in points:
    icon = AwesomeIcon(name=icon_name, marker_color=color, icon_color="white")
    m.add(Marker(location=location, title=name, icon=icon))
m
```


## Next

In [02 — Adding a GeoJSON Layer](./02-Add_GeoJSON.ipynb), we move from individual markers to rendering whole FeatureCollections in a single call.